# Leave-one-subject-out (LOSO): protocol-matched rows for the scoreboard

Three of the scoreboard's comparators used leave-one-subject-out cross-validation; every number of ours uses
5-fold cross-validation by patient. This notebook reruns the two cheap models under LOSO, as pre-registered in
[`docs/PREREG_2026-11-17.md`](docs/PREREG_2026-11-17.md) (Parts 1 and 2). The GAT (Part 3) has its own notebook.

| task | model here | comparator (LOSO) |
|---|---|---|
| AD vs HC, 65 patients | band-power logistic regression, balanced class weights | Wang et al. 2026: AUC 0.912, 87.7 %; benchmark [3]: 0.883, 84.6 % |
| AD vs FTD, 59 patients | C3 lagged cross-correlation on ds006036 (Experiment 10), nested LOSO | Mlinarič et al. 2026: AUC 0.651 |

- **The 5 × 5 numbers stay primary.** LOSO adds a second row, labelled as such, whatever it shows.
- **Why balanced weights.** Leaving one patient out tilts the training classes against that patient's label in
  every fold, which biases a pooled LOSO AUC downward. Cell 4 measures that effect (secondary a).
- **Identity checks.** Cell 4 reproduces the 5 × 5 band-power floor of `results_gat/statistics.json`, and Cell 6
  reproduces Experiment 10's C3 scores, so only the split (and the weights) differ.
- **Reads** `subject_cache/`, `results_gat/statistics.json` and `experiments_ad_vs_ftd/` (features and
  Experiment 10's results). **Writes** `results_loso/`.
- About 5 min, plus ~15 min for the permutation test in Cell 7 (CPU).

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

WORK_DIR  = Path.cwd()
CACHE_DIR = WORK_DIR / "subject_cache"
EXP_DIR   = WORK_DIR / "experiments_ad_vs_ftd"
OUT_DIR   = WORK_DIR / "results_loso"
assert (CACHE_DIR / "sub-001.npz").exists(), "no subject_cache/: open this notebook from the project root"
OUT_DIR.mkdir(exist_ok=True)

SEED   = 42
N_BOOT = 2000
N_PERM = 1000                          # label shuffles through the whole C3 LOSO pipeline (Cell 7)
N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 2))

# ---- AD vs HC: the cache and Cell 5 of GAT_ds004504.ipynb -------------------
N_SUBJECTS, N_AD   = 65, 36            # sub-001..036 AD (label 1), sub-037..065 HC (label 0)
N_FOLDS, N_REPEATS = 5, 5              # the 5 x 5 folds: identity check and secondary (b)

# ---- AD vs FTD: Experiment 10, unchanged --------------------------------------
HC, AD, FTD = 0, 1, 2
INNER_FOLDS = 5
C_GRID      = [0.001, 0.01, 0.1, 1.0]

COMPARATORS = {"AD vs HC": {"Wang et al. 2026 (LOSO)": dict(auc=0.912, acc=0.877),
                            "benchmark [3] (LOSO)":    dict(auc=0.883, acc=0.846)},
               "AD vs FTD": {"Mlinaric et al. 2026 (nested LOSO)": dict(auc=0.651)}}


def verdict(bar, lo, hi):
    """The scoreboard's rule: level when the published value lies inside our 95 % CI."""
    return "level" if lo <= bar <= hi else "ahead" if bar < lo else "behind"


print(f"output : {OUT_DIR}\nworkers: {N_JOBS}")


output : C:\Projects\Major Project\results_loso
workers: 8


In [2]:
# ===========================================================================
# CELL 2 — PROTOCOL, WRITTEN BEFORE ANY MODEL IS FITTED
# ===========================================================================
PROTOCOL = {
    "prereg": "docs/PREREG_2026-11-17.md, Parts 1 and 2 (Part 3, the GAT, is GAT_loso_ds004504.ipynb)",
    "status": "the 5 x 5 numbers stay primary; LOSO is a second row labelled 'LOSO (protocol-matched)', reported "
              "whatever it shows; the better of the two is never chosen",
    "loso": "each patient of the task is the test set once; the model is fitted on the task's other patients",
    "metrics": "patient AUC (Mann-Whitney); accuracy, recall and specificity at 0.5; 95 % CI from 2000 bootstrap "
               "resamples of patients, each class separately, seed 42; nothing tuned on test patients",
    "verdict": "level if the published value lies inside our 95 % CI, ahead if below it, behind if above it",
    "part1_ad_vs_hc": {
        "data": "subject_cache, 65 patients, 95 band-power numbers per 20 s window; a patient's score is the mean "
                "of their window probabilities (Cell 5 of GAT_ds004504.ipynb)",
        "primary": "StandardScaler -> L2 logistic regression, C 0.1, class_weight balanced, max_iter 5000, fitted on "
                   "all windows of the other 64 patients",
        "identity_check": "Cell 5's unweighted model on the 5 x 5 folds reproduces results_gat/statistics.json "
                          "'LR floor' AUC and accuracy",
        "secondary": {"a": "the unweighted model under LOSO (primary - a = the prior-shift effect)",
                      "b": "the balanced model on the 5 x 5 folds (primary - b = the effect of the split)",
                      "tests": "paired patient bootstrap, 95 % CI, descriptive, no Holm"},
        "comparators": "Wang et al. 2026 AUC 0.912 / accuracy 87.7 %; benchmark [3] 0.883 / 84.6 % (context; the "
                       "scoreboard row is the GAT, Part 3)",
    },
    "part2_ad_vs_ftd": {
        "data": "experiments_ad_vs_ftd/features/c3_ds006036_matrices.npz (Experiment 10), unchanged: ds006036, "
                "44 s per patient, 36 AD vs 23 FTD",
        "pipeline": "Experiment 10's run_fold unchanged; outer: leave one of the 59 patients out; C in "
                    "[0.001, 0.01, 0.1, 1] by 5-fold inner stratified CV AUC on the other 58 (nested LOSO); "
                    "inner seed 42 + fold number (1-59)",
        "identity_check": "run_fold on Experiment 10's folds reproduces Experiment 10's ds006036 scores exactly",
        "primary": "AUC of q_AD over the 59 patients with its CI; verdict against Mlinaric et al. 2026 (0.651)",
        "secondary": {"a": "chance under LOSO: 1000 shuffles of the 59 diagnoses through the whole LOSO pipeline; "
                           "one-sided p, median and 95 % range of the shuffled AUCs",
                      "b": "C3 on ds004504, 300 s (Experiment 9's matrices), same LOSO; the recording Mlinaric et "
                           "al. used, but C3 was picked on it, so post hoc and no verdict"},
    },
    "expectation": "LOSO trains on more patients (64 vs 52 for the regression, ~51 vs ~42 for the GAT): a small "
                   "rise of about +0.01-0.02 AUC, verdicts unchanged; a lower number is reported the same way",
}

_path = OUT_DIR / "protocol.json"
if _path.exists():                    # a re-run keeps the original date and must not change the design
    _old = json.loads(_path.read_text(encoding="utf-8"))
    assert {k: v for k, v in _old.items() if k != "written"} == PROTOCOL, "protocol changed after it was written"
    PROTOCOL = _old
else:
    PROTOCOL = {"written": time.strftime("%Y-%m-%d %H:%M") + ", before any model was fitted", **PROTOCOL}
    _path.write_text(json.dumps(PROTOCOL, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"protocol: {_path}  (written {PROTOCOL['written']})")


protocol: C:\Projects\Major Project\results_loso\protocol.json  (written 2026-11-17 14:56, before any model was fitted)


In [3]:
# ===========================================================================
# CELL 3 — AD VS HC: BAND POWER, THE 5 x 5 FOLDS AND THE LOSO FOLDS
# ===========================================================================
bp_parts, y_parts, sid_parts = [], [], []
for i in range(1, N_SUBJECTS + 1):
    sid = f"sub-{i:03d}"
    with np.load(CACHE_DIR / f"{sid}.npz") as z:      # only bp and y are read
        bp_parts.append(z["bp"]); y_parts.append(z["y"])
    sid_parts.append(np.full(len(y_parts[-1]), sid))
    assert int(y_parts[-1][0]) == (1 if i <= N_AD else 0), f"{sid}: wrong label"
BP   = np.concatenate(bp_parts).astype(np.float32)   # as in GAT_ds004504.ipynb
Y    = np.concatenate(y_parts).astype(np.int64)
SID  = np.concatenate(sid_parts)
FEAT = BP.reshape(len(BP), -1)
del bp_parts, y_parts, sid_parts


def make_splits(n_folds=N_FOLDS, n_repeats=N_REPEATS, seed=SEED):
    """Cell 4 of GAT_ds004504.ipynb (subject level), unchanged. The band-power model is fitted on
    train + validation, in that order, as in its Cell 5."""
    splits, idx = [], np.arange(len(Y))
    for r in range(1, n_repeats + 1):
        rs = seed + 1000 * r
        outer = StratifiedGroupKFold(n_folds, shuffle=True, random_state=rs).split(idx, Y, groups=SID)
        for f, (trainval, test) in enumerate(outer, start=1):
            inner = StratifiedKFold(5, shuffle=True, random_state=rs + f)
            subs = np.unique(SID[trainval])
            labs = np.array([Y[SID == s][0] for s in subs])
            _, va = next(inner.split(subs, labs))
            is_val = np.isin(SID[trainval], subs[va])
            splits.append(dict(repeat=r, fold=f, fit=np.concatenate([trainval[~is_val], trainval[is_val]]),
                               test=test))
    return splits


SPLITS_5x5 = make_splits()
SUBJECTS   = np.unique(SID)                                    # sub-001 ... sub-065
SPLITS_LOSO = [dict(repeat=1, fold=k, fit=np.flatnonzero(SID != s), test=np.flatnonzero(SID == s))
               for k, s in enumerate(SUBJECTS, start=1)]
for name, sp in (("5 x 5", SPLITS_5x5), ("LOSO", SPLITS_LOSO)):
    for s in sp:
        assert not set(SID[s["fit"]]) & set(SID[s["test"]]), "SUBJECT LEAKAGE"
    print(f"{name:6s}: {len(sp)} folds; patients fitted per fold "
          f"{sorted({len(np.unique(SID[s['fit']])) for s in sp})}")
print(f"{len(Y):,} windows, {len(SUBJECTS)} patients (AD {N_AD}, HC {N_SUBJECTS - N_AD})")


5 x 5 : 25 folds; patients fitted per fold [52]
LOSO  : 65 folds; patients fitted per fold [64]
5,758 windows, 65 patients (AD 36, HC 29)


In [4]:
# ===========================================================================
# CELL 4 — AD VS HC: THE BAND-POWER MODELS (identity check first)
# ===========================================================================
def fit_predict(fit, test, balanced):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000,
                                                             class_weight="balanced" if balanced else None))
    return clf.fit(FEAT[fit], Y[fit]).predict_proba(FEAT[test])[:, 1]


def run(splits, balanced):
    """Out-of-fold P(AD) per window, one row per repeat."""
    oof = np.full((max(s["repeat"] for s in splits), len(Y)), np.nan)
    probs = Parallel(n_jobs=N_JOBS)(delayed(fit_predict)(s["fit"], s["test"], balanced) for s in splits)
    for s, p in zip(splits, probs):
        oof[s["repeat"] - 1, s["test"]] = p
    assert not np.isnan(oof).any()
    return oof


# ---- Cell 11's statistics, unchanged ---------------------------------------
_subs, _inv = np.unique(SID, return_inverse=True)
S_Y = np.array([Y[_inv == k][0] for k in range(len(_subs))])
_pos, _neg = np.where(S_Y == 1)[0], np.where(S_Y == 0)[0]
_rng  = np.random.default_rng(SEED)
_bpos = _pos[_rng.integers(0, len(_pos), (N_BOOT, len(_pos)))]
_bneg = _neg[_rng.integers(0, len(_neg), (N_BOOT, len(_neg)))]


def to_subjects(oof):
    """(repeats, windows) P(AD) -> (repeats, subjects): mean over each subject's windows."""
    return np.stack([np.bincount(_inv, weights=o) / np.bincount(_inv) for o in oof])


def auc(pos, neg):
    """AUC along the last axis: the chance that a random positive outscores a random negative, ties 1/2."""
    d = pos[..., :, None] - neg[..., None, :]
    return (d > 0).mean((-2, -1)) + 0.5 * (d == 0).mean((-2, -1))


def metrics(sp, pos=_pos, neg=_neg):
    """Subject-level metrics averaged over repeats; pos/neg 1-D (real) or 2-D (one resample per row)."""
    hit = (sp >= 0.5) == S_Y
    sens, spec = hit[:, pos].mean(-1), hit[:, neg].mean(-1)
    n_pos, n_neg = pos.shape[-1], neg.shape[-1]
    return dict(auc=auc(sp[:, pos], sp[:, neg]).mean(0), sens=sens.mean(0), spec=spec.mean(0),
                acc=((sens * n_pos + spec * n_neg) / (n_pos + n_neg)).mean(0))


def boot_p(d):
    return min(1.0, 2 * (min((d <= 0).sum(), (d >= 0).sum()) + 1) / (len(d) + 1))


t0 = time.time()
REF = "5 x 5, unweighted (Cell 5 of GAT_ds004504)"
HC_OOF = {REF: run(SPLITS_5x5, balanced=False)}
_ref = json.loads((WORK_DIR / "results_gat" / "statistics.json").read_text())["models"]["LR floor"]
_m = metrics(to_subjects(HC_OOF[REF]))
assert all(abs(_m[k] - _ref[k]["value"]) < 1e-9 for k in ("auc", "acc", "sens", "spec")), (_m, _ref)
print(f"identity check passed: the 5 x 5 band-power floor reproduces results_gat/statistics.json "
      f"(AUC {_m['auc']:.4f}, accuracy {_m['acc']:.4f})")

PRIMARY_HC = "LOSO, balanced (primary)"
HC_OOF[PRIMARY_HC]                  = run(SPLITS_LOSO, balanced=True)
HC_OOF["LOSO, unweighted (a)"]      = run(SPLITS_LOSO, balanced=False)
HC_OOF["5 x 5, balanced (b)"]       = run(SPLITS_5x5, balanced=True)
print(f"{len(HC_OOF)} band-power runs in {time.time() - t0:.0f} s")


identity check passed: the 5 x 5 band-power floor reproduces results_gat/statistics.json (AUC 0.8724, accuracy 0.7692)


4 band-power runs in 5 s


In [5]:
# ===========================================================================
# CELL 5 — AD VS HC: RESULTS, CIs, VERDICTS AND THE TWO EFFECTS
# ===========================================================================
SUBJ = {k: to_subjects(v) for k, v in HC_OOF.items()}
REAL = {k: metrics(v) for k, v in SUBJ.items()}
BOOT = {k: metrics(v, _bpos, _bneg) for k, v in SUBJ.items()}
lo_hi = lambda v: (np.percentile(v, 2.5), np.percentile(v, 97.5))

rows = []
for k in HC_OOF:
    r = dict(model=k)
    for m in ("auc", "acc", "sens", "spec"):
        r[m] = REAL[k][m]
        r[f"{m}_lo"], r[f"{m}_hi"] = lo_hi(BOOT[k][m])
    rows.append(r)
HC_TABLE = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 220)
print("AD vs HC, band-power logistic regression, 65 patients (5 x 5 rows: mean over 5 repeats)\n")
for k, r in HC_TABLE.iterrows():
    print(f"  {k:44s} AUC {r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}]   accuracy {r.acc:.3f} "
          f"[{r.acc_lo:.3f}, {r.acc_hi:.3f}]   recall {r.sens:.3f}   specificity {r.spec:.3f}")

HC_VERDICTS = []
_p = HC_TABLE.loc[PRIMARY_HC]
for name, bar in COMPARATORS["AD vs HC"].items():
    for m in ("auc", "acc"):
        HC_VERDICTS.append(dict(comparator=name, metric=m, published=bar[m], ours=_p[m],
                                lo=_p[f"{m}_lo"], hi=_p[f"{m}_hi"], verdict=verdict(bar[m], _p[f"{m}_lo"], _p[f"{m}_hi"])))
HC_VERDICTS = pd.DataFrame(HC_VERDICTS)
print(f"\nverdicts for '{PRIMARY_HC}' (context: the scoreboard row is the GAT, Part 3)")
print(HC_VERDICTS.round(3).to_string(index=False))

EFFECTS = []
for a, b, what in ((PRIMARY_HC, "LOSO, unweighted (a)", "prior-shift effect (balanced vs unweighted, both LOSO)"),
                   (PRIMARY_HC, "5 x 5, balanced (b)", "split effect (LOSO vs 5 x 5, both balanced)")):
    for m in ("auc", "acc"):
        d = BOOT[a][m] - BOOT[b][m]
        EFFECTS.append(dict(effect=what, metric=m, diff=REAL[a][m] - REAL[b][m], lo=lo_hi(d)[0], hi=lo_hi(d)[1],
                            p=boot_p(d)))
EFFECTS = pd.DataFrame(EFFECTS)
print("\npaired differences, primary minus secondary (descriptive, no Holm)")
print(EFFECTS.round(3).to_string(index=False))


AD vs HC, band-power logistic regression, 65 patients (5 x 5 rows: mean over 5 repeats)

  5 x 5, unweighted (Cell 5 of GAT_ds004504)   AUC 0.872 [0.787, 0.941]   accuracy 0.769 [0.674, 0.849]   recall 0.772   specificity 0.766
  LOSO, balanced (primary)                     AUC 0.897 [0.810, 0.959]   accuracy 0.769 [0.662, 0.862]   recall 0.806   specificity 0.724
  LOSO, unweighted (a)                         AUC 0.894 [0.807, 0.957]   accuracy 0.769 [0.662, 0.862]   recall 0.806   specificity 0.724
  5 x 5, balanced (b)                          AUC 0.873 [0.788, 0.941]   accuracy 0.769 [0.674, 0.849]   recall 0.772   specificity 0.766

verdicts for 'LOSO, balanced (primary)' (context: the scoreboard row is the GAT, Part 3)
             comparator metric  published  ours    lo    hi verdict
Wang et al. 2026 (LOSO)    auc      0.912 0.897 0.810 0.959   level
Wang et al. 2026 (LOSO)    acc      0.877 0.769 0.662 0.862  behind
   benchmark [3] (LOSO)    auc      0.883 0.897 0.810 0.959  

---
## AD vs FTD: C3 under nested LOSO (Cells 6–7)

Experiment 10's C3 pipeline, unchanged, with leave-one-patient-out as the outer loop. Each of the 59 patients is
scored by a model whose C was chosen by 5-fold inner CV on the other 58, as in Mlinarič et al. 2026.

In [6]:
# ===========================================================================
# CELL 6 — AD VS FTD: C3 UNDER NESTED LOSO (identity check first)
# ===========================================================================
subjects = pd.read_csv(EXP_DIR / "features" / "subjects.csv")
assert list(subjects.participant_id) == [f"sub-{i:03d}" for i in range(1, 89)]
LABELS   = subjects.label.to_numpy()
DEMENTIA = np.flatnonzero(LABELS != HC)
with np.load(EXP_DIR / "features" / "c3_ds006036_matrices.npz") as z:
    XCZ_06 = z["xcz_06"]                                     # ds006036, 44 s (Experiment 10)
with np.load(EXP_DIR / "features" / "riemann_matrices.npz") as z:
    XCZ_04 = z["xcz"]                                        # ds004504, 300 s (Experiment 9)
_e10 = np.load(EXP_DIR / "results" / "c3_ds006036" / "c3_oof.npz", allow_pickle=True)
assert list(_e10["subjects"]) == list(subjects.participant_id) and (_e10["labels"] == LABELS).all()


def run_fold(X, test, labels, seed):
    """Experiment 10's C3, unchanged: tuned on the dementia patients outside `test`, q_AD for every test patient."""
    train = np.setdiff1d(np.flatnonzero(labels != HC), test)
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
    est = GridSearchCV(pipe, {"logisticregression__C": C_GRID}, cv=inner, scoring="roc_auc")
    return est.fit(X[train], (labels[train] == AD).astype(int)).predict_proba(X[test])[:, 1]


# ---- identity check: Experiment 10's folds and seeds give Experiment 10's scores ----
_fold_of = _e10["fold_of"]
_q = np.full(_fold_of.shape, np.nan)
for r in range(_fold_of.shape[0]):
    for f in range(1, N_FOLDS + 1):
        test = np.flatnonzero(_fold_of[r] == f)
        _q[r, test] = run_fold(XCZ_06, test, LABELS, SEED + 100 * (r + 1) + f)
_q10 = _e10["q_ad"][list(_e10["runs"]).index("ds006036, 44 s (primary)")]
assert np.allclose(_q, _q10, atol=1e-10), "the pipeline differs from Experiment 10's"
print("identity check passed: Experiment 10's C3 scores on ds006036 reproduced exactly")


def run_loso(X, labels):
    """q_AD for each of the 59 dementia patients, from a model fitted (and tuned) on the other 58."""
    q = np.full(len(labels), np.nan)
    for k, i in enumerate(DEMENTIA, start=1):
        q[i] = run_fold(X, np.array([i]), labels, SEED + k)[0]
    return q


t0 = time.time()
PRIMARY_FTD = "ds006036, 44 s (primary)"
Q_LOSO = {PRIMARY_FTD: run_loso(XCZ_06, LABELS),
          "ds004504, 300 s (b, post hoc)": run_loso(XCZ_04, LABELS)}
print(f"nested LOSO done in {time.time() - t0:.0f} s")

YD = (LABELS[DEMENTIA] == AD).astype(int)
POS, NEG = np.flatnonzero(YD == 1), np.flatnonzero(YD == 0)
_rng = np.random.default_rng(SEED)                           # Experiment 10's bootstrap
_bp  = _rng.choice(POS, (N_BOOT, len(POS)))
_bn  = _rng.choice(NEG, (N_BOOT, len(NEG)))

rows = []
for n, q in Q_LOSO.items():
    qd = q[DEMENTIA]
    call = qd >= 0.5
    rec_ad, rec_ftd = call[POS].mean(), (~call[NEG]).mean()
    boot = auc(qd[_bp], qd[_bn])
    rows.append(dict(run=n, auc=auc(qd[POS], qd[NEG]), auc_lo=np.percentile(boot, 2.5),
                     auc_hi=np.percentile(boot, 97.5), bal_acc=(rec_ad + rec_ftd) / 2,
                     recall_AD=rec_ad, recall_FTD=rec_ftd))
FTD_TABLE = pd.DataFrame(rows).set_index("run")
_s10 = json.loads((EXP_DIR / "results" / "c3_ds006036" / "summary.json").read_text())
print("\nC3, AD vs FTD, 59 patients, nested LOSO\n")
print(FTD_TABLE.round(3).to_string())
print(f"\nfor reference, 5 x 5 (Experiment 10): ds006036 {_s10['primary']['auc']:.3f} {_s10['primary']['ci']}; "
      f"ds004504 300 s (Experiment 9, post hoc) 0.628")

_p = FTD_TABLE.loc[PRIMARY_FTD]
FTD_VERDICTS = pd.DataFrame([dict(comparator=n, metric="auc", published=b["auc"], ours=_p.auc, lo=_p.auc_lo,
                                  hi=_p.auc_hi, verdict=verdict(b["auc"], _p.auc_lo, _p.auc_hi))
                             for n, b in COMPARATORS["AD vs FTD"].items()])
print("\n" + FTD_VERDICTS.round(3).to_string(index=False))


identity check passed: Experiment 10's C3 scores on ds006036 reproduced exactly


nested LOSO done in 19 s

C3, AD vs FTD, 59 patients, nested LOSO

                                 auc  auc_lo  auc_hi  bal_acc  recall_AD  recall_FTD
run                                                                                 
ds006036, 44 s (primary)       0.574   0.412   0.719    0.572      0.667       0.478
ds004504, 300 s (b, post hoc)  0.678   0.524   0.809    0.636      0.750       0.522

for reference, 5 x 5 (Experiment 10): ds006036 0.641 [0.5152, 0.7527]; ds004504 300 s (Experiment 9, post hoc) 0.628

                        comparator metric  published  ours    lo    hi verdict
Mlinaric et al. 2026 (nested LOSO)    auc      0.651 0.574 0.412 0.719   level


In [7]:
# ===========================================================================
# CELL 7 — SECONDARY (a): CHANCE UNDER LOSO (the whole pipeline on shuffled diagnoses)
# ===========================================================================
def shuffled_auc(k):
    labels = LABELS.copy()
    labels[DEMENTIA] = np.random.default_rng(SEED + 10_000 + k).permutation(labels[DEMENTIA])
    q = run_loso(XCZ_06, labels)[DEMENTIA]
    yd = labels[DEMENTIA] == AD
    return auc(q[yd], q[~yd])


t0 = time.time()
NULL = np.array(Parallel(n_jobs=N_JOBS)(delayed(shuffled_auc)(k) for k in range(N_PERM)))
OBS = FTD_TABLE.loc[PRIMARY_FTD, "auc"]
P_PERM = (1 + (NULL >= OBS).sum()) / (1 + len(NULL))
print(f"{len(NULL)} shuffles in {time.time() - t0:.0f} s")
print(f"C3 LOSO AUC {OBS:.3f}; shuffled diagnoses: median {np.median(NULL):.3f}, 95 % range "
      f"{np.percentile(NULL, 2.5):.3f}-{np.percentile(NULL, 97.5):.3f}; one-sided p = {P_PERM:.4f}")


1000 shuffles in 3037 s
C3 LOSO AUC 0.574; shuffled diagnoses: median 0.488, 95 % range 0.285-0.686; one-sided p = 0.2068


In [8]:
# ===========================================================================
# CELL 8 — SAVE, AND THE ROWS FOR THE SCOREBOARD
# ===========================================================================
np.savez_compressed(OUT_DIR / "bandpower_oof.npz", subjects=SUBJECTS, labels=S_Y, runs=np.array(list(HC_OOF)),
                    **{f"window_p_{i}": v for i, v in enumerate(HC_OOF.values())},
                    subject_p=np.array([SUBJ[k].mean(0) for k in HC_OOF]))
np.savez_compressed(OUT_DIR / "c3_oof.npz", subjects=subjects.participant_id.to_numpy(), labels=LABELS,
                    runs=np.array(list(Q_LOSO)), q_ad=np.stack([Q_LOSO[n] for n in Q_LOSO]))
HC_TABLE.to_csv(OUT_DIR / "ad_vs_hc_bandpower.csv")
EFFECTS.to_csv(OUT_DIR / "ad_vs_hc_effects.csv", index=False)
FTD_TABLE.to_csv(OUT_DIR / "ad_vs_ftd_c3.csv")
pd.concat([HC_VERDICTS.assign(task="AD vs HC, band power"), FTD_VERDICTS.assign(task="AD vs FTD, C3")]).to_csv(
    OUT_DIR / "verdicts.csv", index=False)
pd.DataFrame(dict(shuffled_auc=NULL)).to_csv(OUT_DIR / "c3_permutation.csv", index=False)

_h, _f = HC_TABLE.loc[PRIMARY_HC], FTD_TABLE.loc[PRIMARY_FTD]
SUMMARY = dict(
    protocol=PROTOCOL["written"],
    ad_vs_hc_bandpower=dict(auc=round(_h.auc, 4), auc_ci=[round(_h.auc_lo, 4), round(_h.auc_hi, 4)],
                            acc=round(_h.acc, 4), acc_ci=[round(_h.acc_lo, 4), round(_h.acc_hi, 4)],
                            recall=round(_h.sens, 4), specificity=round(_h.spec, 4),
                            five_by_five_unweighted=dict(auc=round(HC_TABLE.loc[REF, "auc"], 4),
                                                         acc=round(HC_TABLE.loc[REF, "acc"], 4)),
                            effects=EFFECTS.round(4).to_dict(orient="records"),
                            verdicts=HC_VERDICTS.round(4).to_dict(orient="records")),
    ad_vs_ftd_c3=dict(auc=round(_f.auc, 4), auc_ci=[round(_f.auc_lo, 4), round(_f.auc_hi, 4)],
                      bal_acc=round(_f.bal_acc, 4), recall_AD=round(_f.recall_AD, 4),
                      recall_FTD=round(_f.recall_FTD, 4), p_permutation=round(P_PERM, 4), n_perm=N_PERM,
                      null_median=round(float(np.median(NULL)), 4),
                      null_95=[round(float(np.percentile(NULL, q)), 4) for q in (2.5, 97.5)],
                      ds004504_300s_post_hoc=dict(auc=round(FTD_TABLE.iloc[1].auc, 4),
                                                  auc_ci=[round(FTD_TABLE.iloc[1].auc_lo, 4),
                                                          round(FTD_TABLE.iloc[1].auc_hi, 4)]),
                      five_by_five=_s10["primary"]["auc"],
                      verdicts=FTD_VERDICTS.round(4).to_dict(orient="records")))
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float),
                                      encoding="utf-8")

print("LOSO rows (protocol-matched; the 5 x 5 rows stay primary)\n")
print(f"  AD vs HC, band power : AUC {_h.auc:.3f} [{_h.auc_lo:.3f}, {_h.auc_hi:.3f}], accuracy {_h.acc:.3f}   "
      f"(5 x 5: {HC_TABLE.loc[REF, 'auc']:.3f} / {HC_TABLE.loc[REF, 'acc']:.3f})")
print(f"  AD vs FTD, C3        : AUC {_f.auc:.3f} [{_f.auc_lo:.3f}, {_f.auc_hi:.3f}], permutation p {P_PERM:.4f}   "
      f"(5 x 5: {_s10['primary']['auc']:.3f})")
print(f"\nsaved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


LOSO rows (protocol-matched; the 5 x 5 rows stay primary)

  AD vs HC, band power : AUC 0.897 [0.810, 0.959], accuracy 0.769   (5 x 5: 0.872 / 0.769)
  AD vs FTD, C3        : AUC 0.574 [0.412, 0.719], permutation p 0.2068   (5 x 5: 0.641)

saved to C:\Projects\Major Project\results_loso: ad_vs_ftd_c3.csv, ad_vs_hc_bandpower.csv, ad_vs_hc_effects.csv, bandpower_oof.npz, c3_oof.npz, c3_permutation.csv, protocol.json, summary.json, verdicts.csv
